# AT URI Resolution

This notebook builds AT URI parsing and resolution, AT URIs are the addressing scheme used by ATProto to identify records. They have the form
addresses records. AT URIs are the addressing scheme for ATProto records:
`at://authority/collection/rkey`.

**What you'll learn:**
- AT URI structure and component parsing
- Round-trip URI construction
- Resolving URIs against a record store

**Estimated Time:** 15-20 minutes

## Step 1: AT URI Class

An AT URI has three components: authority (DID or handle), collection (NSID),
and record key (rkey). record retrieval resolves these
components to find the actual record.

In [ ]:
@interface ATURI : NSObject
@property (nonatomic, strong) NSString *authority;
@property (nonatomic, strong) NSString *collection;
@property (nonatomic, strong) NSString *rkey;
- (instancetype)initWithString:(NSString *)uri;
- (NSString *)stringValue;
- (NSString *)description;
@end

@implementation ATURI
- (instancetype)initWithString:(NSString *)uri {
    self = [super init];
    if (self) {
        // Strip "at://" prefix
        NSString *rest = uri;
        if ([rest hasPrefix:@"at://"]) {
            rest = [rest substringFromIndex:5];
        }
        // Split into components
        NSArray *parts = [rest componentsSeparatedByString:@"/"];
        if ([parts count] >= 1) _authority = [parts objectAtIndex:0];
        if ([parts count] >= 2) _collection = [parts objectAtIndex:1];
        if ([parts count] >= 3) _rkey = [parts objectAtIndex:2];
    }
    return self;
}
- (NSString *)stringValue {
    return [NSString stringWithFormat:@"at://%@/%@/%@", _authority, _collection, _rkey];
}
- (NSString *)description {
    return [NSString stringWithFormat:@"ATURI(authority=%@, collection=%@, rkey=%@)", _authority, _collection, _rkey];
}
@end

// Parse some URIs
ATURI *uri1 = [[ATURI alloc] initWithString:@"at://did:plc:abc123/app.bsky.feed.post/3kabc"];
NSLog(@"%@", [uri1 description]);

ATURI *uri2 = [[ATURI alloc] initWithString:@"at://alice.bsky.social/app.bsky.actor.profile/self"];
NSLog(@"%@", [uri2 description]);

// Round-trip
NSLog(@"Round-trip: %@", [uri1 stringValue]);

## Step 2: Record Store (inline)

We need a record store to resolve against. This is a simplified version of
services — keyed by AT URI.

In [ ]:
@interface RecordStore : NSObject
@property (nonatomic, strong) NSMutableDictionary *records;
- (void)putRecord:(NSDictionary *)record atURI:(NSString *)uri;
- (NSDictionary *)getRecordAtURI:(NSString *)uri;
@end

@implementation RecordStore
- (instancetype)init {
    self = [super init];
    if (self) { _records = [NSMutableDictionary dictionary]; }
    return self;
}
- (void)putRecord:(NSDictionary *)record atURI:(NSString *)uri {
    [_records setObject:record forKey:uri];
    NSLog(@"Stored at %@", uri);
}
- (NSDictionary *)getRecordAtURI:(NSString *)uri {
    NSDictionary *record = [_records objectForKey:uri];
    if (record == nil) {
        NSLog(@"Not found: %@", uri);
    }
    return record;
}
@end

## Step 3: AT URI Resolver

The resolver takes an AT URI, reconstructs the full URI string, and looks it up
in the record store. This mirrors record retrieval.

In [ ]:
@interface ATURIResolver : NSObject
@property (nonatomic, strong) RecordStore *store;
- (NSDictionary *)resolveURI:(ATURI *)uri;
- (NSDictionary *)resolveString:(NSString *)uriString;
@end

@implementation ATURIResolver
- (instancetype)initWithStore:(RecordStore *)store {
    self = [super init];
    if (self) { _store = store; }
    return self;
}
- (NSDictionary *)resolveURI:(ATURI *)uri {
    NSString *uriString = [uri stringValue];
    NSDictionary *record = [_store getRecordAtURI:uriString];
    if (record != nil) {
        NSLog(@"Resolved %@ -> %@", [uri description], record);
    }
    return record;
}
- (NSDictionary *)resolveString:(NSString *)uriString {
    ATURI *uri = [[ATURI alloc] initWithString:uriString];
    return [self resolveURI:uri];
}
@end

## Step 4: Full Resolution Demo

Create records, build AT URIs, resolve them, verify round-trip.

In [ ]:
RecordStore *store = [[RecordStore alloc] init];
ATURIResolver *resolver = [[ATURIResolver alloc] initWithStore:store];

// Store some records
[store putRecord:@{@"text": @"Hello ATProto!", @"createdAt": @"2025-01-01"}
         atURI:@"at://did:plc:abc123/app.bsky.feed.post/3kabc"];

[store putRecord:@{@"displayName": @"Alice", @"description": @"Test user"}
         atURI:@"at://did:plc:abc123/app.bsky.actor.profile/self"];

// Resolve by constructing ATURI
ATURI *postUri = [[ATURI alloc] initWithString:@"at://did:plc:abc123/app.bsky.feed.post/3kabc"];
NSDictionary *post = [resolver resolveURI:postUri];
NSLog(@"Post text: %@", [post objectForKey:@"text"]);

// Resolve by string
NSDictionary *profile = [resolver resolveString:@"at://did:plc:abc123/app.bsky.actor.profile/self"];
NSLog(@"Profile name: %@", [profile objectForKey:@"displayName"]);

// Non-existent record
NSDictionary *missing = [resolver resolveString:@"at://did:plc:xyz/app.bsky.feed.post/missing"];
NSLog(@"Missing: %@", missing);

## Exercise

Add a `collectionURI` method to `ATURI` that returns just the collection part of the URI
(`at://authority/collection`) without the rkey. This is useful for listing all records
in a collection.

Hint: reconstruct the URI without the rkey component.

In [ ]:
// Exercise: implement collectionURI
// Your code here...

## Solution

In [ ]:
// Solution: collectionURI returns at://authority/collection
@interface ATURI2 : NSObject
@property (nonatomic, strong) NSString *authority;
@property (nonatomic, strong) NSString *collection;
@property (nonatomic, strong) NSString *rkey;
- (instancetype)initWithString:(NSString *)uri;
- (NSString *)collectionURI;
@end

@implementation ATURI2
- (instancetype)initWithString:(NSString *)uri {
    self = [super init];
    if (self) {
        NSString *rest = uri;
        if ([rest hasPrefix:@"at://"]) rest = [rest substringFromIndex:5];
        NSArray *parts = [rest componentsSeparatedByString:@"/"];
        if ([parts count] >= 1) _authority = [parts objectAtIndex:0];
        if ([parts count] >= 2) _collection = [parts objectAtIndex:1];
        if ([parts count] >= 3) _rkey = [parts objectAtIndex:2];
    }
    return self;
}
- (NSString *)collectionURI {
    return [NSString stringWithFormat:@"at://%@/%@", _authority, _collection];
}
@end

ATURI2 *u = [[ATURI2 alloc] initWithString:@"at://did:plc:abc/app.bsky.feed.post/3k"];
NSLog(@"Collection URI: %@", [u collectionURI]);

## Next Steps

You have learned about ATProto's core data structures and protocols. Continue to explore how these concepts apply when building a PDS.